In [87]:
#!pip install google-api-python-client
#!pip install openai
#!pip install --upgrade openai
#!pip install transformers torch

In [1]:
import googleapiclient.discovery
from datetime import datetime, timedelta
import pandas as pd
from transformers import pipeline  # For Hugging Face Transformers
import os  # For checking if the CSV file exists


# Replace with your own API key
YOUTUBE_API_KEY = 'key'

# Channel handle (used only if CHANNEL_ID is not provided)
CHANNEL_HANDLE = "CNBC-TV18" ## CNBC-TV18, ETNow, moneycontrol

# Channel ID (leave blank to use channel handle)
CHANNEL_ID = ""  

# Set your desired date range (format: YYYY-MM-DD HH:MM:SS)
START_DATE = "2025-03-21 08:00:00"  # Example: Start date with time
END_DATE = "2025-03-21 23:00:00"    # Example: End date with time

# Number of latest videos to fetch (used only if START_DATE is not provided)
MAX_VIDEOS = 50  # Example: Fetch 100 latest videos

# Categories for classification
CATEGORIES = ["Stock_News", "Economy_News", "Political_News", "Entertainment_News", "Other"]

# Load Hugging Face Transformers pipeline for text classification
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

def get_channel_id_and_uploads_playlist(api_key, channel_handle):
    # Initialize the YouTube API client
    youtube = googleapiclient.discovery.build('youtube', 'v3', developerKey=api_key)
    
    # Fetch the channel ID using the channel handle
    request = youtube.search().list(
        part='snippet',
        q=channel_handle,
        type='channel',
        maxResults=1
    )
    response = request.execute()
    
    if response['items']:
        channel_id = response['items'][0]['snippet']['channelId']
        print(f"Channel ID for '{channel_handle}': {channel_id}")
        
        # Fetch the uploads playlist ID
        request = youtube.channels().list(
            part='contentDetails',
            id=channel_id
        )
        response = request.execute()
        
        uploads_playlist_id = response['items'][0]['contentDetails']['relatedPlaylists']['uploads']
        print(f"Uploads Playlist ID: {uploads_playlist_id}")
        
        return channel_id, uploads_playlist_id
    else:
        raise ValueError(f"Channel with handle '{channel_handle}' not found.")

def get_video_details(api_key, video_id):
    # Initialize the YouTube API client
    youtube = googleapiclient.discovery.build('youtube', 'v3', developerKey=api_key)
    
    try:
        # Fetch video details
        request = youtube.videos().list(
            part='contentDetails',
            id=video_id
        )
        response = request.execute()
        
        if response['items']:
            duration = response['items'][0]['contentDetails'].get('duration', 'PT0M')  # Default to 0 minutes if duration is missing
            # Convert ISO 8601 duration to minutes
            minutes = convert_duration_to_minutes(duration)
            return minutes
        else:
            return None
    except Exception as e:
        print(f"Error fetching video details for video ID {video_id}: {e}")
        return None

def convert_duration_to_minutes(duration):
    # Convert ISO 8601 duration (e.g., PT5M30S) to minutes
    import re
    hours = re.search(r'(\d+)H', duration)
    minutes = re.search(r'(\d+)M', duration)
    seconds = re.search(r'(\d+)S', duration)
    
    total_minutes = 0
    if hours:
        total_minutes += int(hours.group(1)) * 60
    if minutes:
        total_minutes += int(minutes.group(1))
    if seconds:
        total_minutes += int(seconds.group(1)) / 60
    
    return round(total_minutes, 2)

def classify_title(title):
    # Use Hugging Face Transformers to classify the title
    result = classifier(title, candidate_labels=CATEGORIES)
    return result["labels"][0]  # Return the top predicted category

def get_video_titles(api_key, uploads_playlist_id, max_results=100, start_date=None, end_date=None):
    # Initialize the YouTube API client
    youtube = googleapiclient.discovery.build('youtube', 'v3', developerKey=api_key)
    
    # Convert start and end dates to datetime objects for comparison (if provided)
    if start_date and end_date:
        start_date = datetime.strptime(start_date, "%Y-%m-%d %H:%M:%S")  # Parse with time
        end_date = datetime.strptime(end_date, "%Y-%m-%d %H:%M:%S")      # Parse with time
        
        # Check if start_date is in the future
        if start_date > datetime.now():
            print("Warning: START_DATE is in the future. No videos will be found unless the date is valid.")
    
    # Fetch the videos from the uploads playlist
    video_data = []
    next_page_token = None
    total_results = 0
    page_count = 0
    
    while True:
        try:
            page_count += 1
            print(f"Fetching page {page_count}...")
            
            # Determine the number of results to fetch per request
            max_results_per_request = 50  # YouTube API allows up to 50 results per request
            
            # If no date range is provided, limit the total results to MAX_VIDEOS
            if not start_date or not end_date:
                remaining_results = max_results - total_results
                if remaining_results <= 0:
                    print("Max results reached. Stopping fetch.")
                    break
                max_results_per_request = min(max_results_per_request, remaining_results)
            
            print(f"Fetching {max_results_per_request} videos per request...")
            
            # Fetch videos
            request = youtube.playlistItems().list(
                part='snippet',
                playlistId=uploads_playlist_id,
                maxResults=max_results_per_request,
                pageToken=next_page_token
            )
            print("Sending API request...")
            response = request.execute()
            print("API request completed.")
            
            # Check if the response contains items
            if 'items' not in response or not response['items']:
                print("No more videos found in the response.")
                break
            
            print(f"Found {len(response['items'])} videos in this page.")
            
            for item in response['items']:
                video_title = item['snippet']['title']
                video_published_at = item['snippet']['publishedAt']
                video_published_at = datetime.strptime(video_published_at, "%Y-%m-%dT%H:%M:%SZ")
                video_id = item['snippet']['resourceId']['videoId']
                
                print(f"Processing video: {video_title} (Published: {video_published_at})")
                
                # Stop fetching if the published date is older than START_DATE
                if start_date and video_published_at < start_date:
                    print(f"Video published before {start_date}. Stopping fetch.")
                    break
                
                # Fetch video length in minutes
                video_length_minutes = get_video_details(api_key, video_id)
                print(f"Video length: {video_length_minutes} minutes")
                
                # Apply date filter if start_date and end_date are provided
                if start_date and end_date:
                    if start_date <= video_published_at < end_date:
                        video_data.append({
                            "Title": video_title,
                            "Published_At": video_published_at.strftime("%Y-%m-%d %H:%M:%S"),
                            "Video_Length_Minutes": video_length_minutes
                        })
                        total_results += 1
                        print(f"Added video to results (Total: {total_results})")
                else:
                    video_data.append({
                        "Title": video_title,
                        "Published_At": video_published_at.strftime("%Y-%m-%d %H:%M:%S"),
                        "Video_Length_Minutes": video_length_minutes
                    })
                    total_results += 1
                    print(f"Added video to results (Total: {total_results})")
            
            # Stop fetching if the published date is older than START_DATE
            if start_date and video_published_at < start_date:
                break
            
            # Check if there are more pages
            next_page_token = response.get('nextPageToken')
            if not next_page_token:
                print("No more pages to fetch.")
                break  # No more pages
            
            print(f"Next page token: {next_page_token}")
        
        except Exception as e:
            print(f"Error fetching videos: {e}")
            break  # Exit the loop if an error occurs
    
    print(f"Total videos fetched: {total_results}")
    return video_data

def main():
    try:
        # Get the uploads playlist ID
        if CHANNEL_ID:
            print(f"Using provided Channel ID: {CHANNEL_ID}")
            youtube = googleapiclient.discovery.build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
            request = youtube.channels().list(
                part='contentDetails',
                id=CHANNEL_ID
            )
            response = request.execute()
            uploads_playlist_id = response['items'][0]['contentDetails']['relatedPlaylists']['uploads']
            print(f"Uploads Playlist ID: {uploads_playlist_id}")
        else:
            print(f"Fetching Channel ID for handle '@{CHANNEL_HANDLE}'...")
            _, uploads_playlist_id = get_channel_id_and_uploads_playlist(YOUTUBE_API_KEY, CHANNEL_HANDLE)
        
        # Fetch videos based on the provided options
        if START_DATE and END_DATE:
            print(f"Fetching videos between {START_DATE} and {END_DATE}...")
            video_data = get_video_titles(YOUTUBE_API_KEY, uploads_playlist_id, start_date=START_DATE, end_date=END_DATE)
            print(f"Found {len(video_data)} videos:")
        else:
            print(f"Fetching {MAX_VIDEOS} latest videos...")
            video_data = get_video_titles(YOUTUBE_API_KEY, uploads_playlist_id, max_results=MAX_VIDEOS)
            print(f"Found {len(video_data)} latest videos:")
        
        # Store video data in a dataframe
        df = pd.DataFrame(video_data)
        
        # Add Channel Handle column
        df["Channel_Handle"] = f"@{CHANNEL_HANDLE}"
        
        # Add empty Category column
        df["Category"] = None  # Initially empty
        
        # Add empty Tag column
        df["Tag"] = None  # Initially empty
        
        # Optional: Classify each title into categories
        # Uncomment the following line to enable classification
        # df["Category"] = df["Title"].apply(classify_title)
        
        # Reorder columns
        df = df[["Channel_Handle", "Title", "Published_At", "Video_Length_Minutes", "Category", "Tag"]]
        
        # Append to CSV file if it exists, otherwise create a new one
        csv_file = "classified_videos.csv"
        if os.path.exists(csv_file):
            df.to_csv(csv_file, mode='a', header=False, index=False)
            print(f"Data appended to '{csv_file}'")
        else:
            df.to_csv(csv_file, index=False)
            print(f"Data saved to '{csv_file}'")
    
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    main()


Fetching Channel ID for handle '@CNBC-TV18'...
Channel ID for 'CNBC-TV18': UCmRbHAgG2k2vDUvb3xsEunQ
Uploads Playlist ID: UUmRbHAgG2k2vDUvb3xsEunQ
Fetching videos between 2025-03-21 08:00:00 and 2025-03-21 23:00:00...
Fetching page 1...
Fetching 50 videos per request...
Sending API request...
API request completed.
Found 50 videos in this page.
Processing video: LIVE | Look Ahead To Tomorrow's Trade: What Are Key Events, Stocks To Watch | Markets Forward (Published: 2025-03-24 10:05:26)
Video length: 0 minutes
Processing video: Which Are The Best Stocks To Buy, Hold & Sell: All Your Stock Queries Answered | CNBC TV18 (Published: 2025-03-24 10:00:12)
Video length: 0 minutes
Processing video: Market Closing Bell LIVE | Rally Continues For The 6th Day; Nifty Above 23600, Sensex Rises 1100 Pts (Published: 2025-03-24 10:00:09)
Video length: 62.78 minutes
Processing video: Nuvama Cuts Target Price Of Jindal Stainless; CFO Anurag Mantri Tenders Resignation | CNBC TV18 (Published: 2025-03-24 0

In [83]:
# Usage limit issue

import googleapiclient.discovery
from datetime import datetime, timedelta
import pandas as pd
import openai  # For ChatGPT API

YOUTUBE_API_KEY = 'key'
OPENAI_API_KEY = 'key'  # For ChatGPT API

# Channel ID (leave blank to use channel handle)
CHANNEL_ID = ""  # Example: CNBC-TV18 channel ID

# Channel handle (used only if CHANNEL_ID is not provided)
CHANNEL_HANDLE = "CNBC-TV18"

# Set your desired date range (format: YYYY-MM-DD)
START_DATE = ""  # Example: Start date
END_DATE = ""    # Example: End date

# Number of latest videos to fetch (used only if START_DATE is not provided)
MAX_VIDEOS = 10  # Example: Fetch 100 latest videos

# Categories for classification
CATEGORIES = ["Stock_News", "Economy_News", "Political_News", "Entertainment_News", "Other"]

def get_channel_id_and_uploads_playlist(api_key, channel_handle):
    # Initialize the YouTube API client
    youtube = googleapiclient.discovery.build('youtube', 'v3', developerKey=api_key)
    
    # Fetch the channel ID using the channel handle
    request = youtube.search().list(
        part='snippet',
        q=channel_handle,
        type='channel',
        maxResults=1
    )
    response = request.execute()
    
    if response['items']:
        channel_id = response['items'][0]['snippet']['channelId']
        print(f"Channel ID for '{channel_handle}': {channel_id}")
        
        # Fetch the uploads playlist ID
        request = youtube.channels().list(
            part='contentDetails',
            id=channel_id
        )
        response = request.execute()
        
        uploads_playlist_id = response['items'][0]['contentDetails']['relatedPlaylists']['uploads']
        print(f"Uploads Playlist ID: {uploads_playlist_id}")
        
        return channel_id, uploads_playlist_id
    else:
        raise ValueError(f"Channel with handle '{channel_handle}' not found.")

def classify_title(title):
    # Use ChatGPT API to classify the title
    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",  # Use GPT-3.5-turbo or GPT-4
        messages=[
            {"role": "system", "content": f"Classify the following news title into one of these categories: {', '.join(CATEGORIES)}. Reply with only the category name."},
            {"role": "user", "content": title}
        ],
        max_tokens=10,
        temperature=0
    )
    category = response.choices[0].message.content.strip()
    return category if category in CATEGORIES else "Other"

def get_video_titles(api_key, uploads_playlist_id, max_results=100, start_date=None, end_date=None):
    # Initialize the YouTube API client
    youtube = googleapiclient.discovery.build('youtube', 'v3', developerKey=api_key)
    
    # Convert start and end dates to datetime objects for comparison (if provided)
    if start_date and end_date:
        start_date = datetime.strptime(start_date, "%Y-%m-%d")
        end_date = datetime.strptime(end_date, "%Y-%m-%d") + timedelta(days=1)  # Include the entire end date
        
        # Check if start_date is in the future
        if start_date > datetime.now():
            print("Warning: START_DATE is in the future. No videos will be found unless the date is valid.")
    
    # Fetch the videos from the uploads playlist
    video_titles = []
    next_page_token = None
    total_results = 0
    
    while True:
        request = youtube.playlistItems().list(
            part='snippet',
            playlistId=uploads_playlist_id,
            maxResults=min(50, max_results - total_results),  # Fetch up to 50 items per request
            pageToken=next_page_token
        )
        response = request.execute()
        
        for item in response['items']:
            video_title = item['snippet']['title']
            video_published_at = item['snippet']['publishedAt']
            video_published_at = datetime.strptime(video_published_at, "%Y-%m-%dT%H:%M:%SZ")
            
            # Apply date filter if start_date and end_date are provided
            if start_date and end_date:
                if start_date <= video_published_at < end_date:
                    video_titles.append((video_title, video_published_at.strftime("%Y-%m-%d %H:%M:%S")))
            else:
                video_titles.append((video_title, video_published_at.strftime("%Y-%m-%d %H:%M:%S")))
            
            total_results += 1
            if total_results >= max_results:
                break
        
        next_page_token = response.get('nextPageToken')
        
        if not next_page_token or total_results >= max_results:
            break
    
    return video_titles

def main():
    try:
        # Get the uploads playlist ID
        if CHANNEL_ID:
            print(f"Using provided Channel ID: {CHANNEL_ID}")
            youtube = googleapiclient.discovery.build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
            request = youtube.channels().list(
                part='contentDetails',
                id=CHANNEL_ID
            )
            response = request.execute()
            uploads_playlist_id = response['items'][0]['contentDetails']['relatedPlaylists']['uploads']
            print(f"Uploads Playlist ID: {uploads_playlist_id}")
        else:
            print(f"Fetching Channel ID for handle '@{CHANNEL_HANDLE}'...")
            _, uploads_playlist_id = get_channel_id_and_uploads_playlist(YOUTUBE_API_KEY, CHANNEL_HANDLE)
        
        # Fetch videos based on the provided options
        if START_DATE and END_DATE:
            print(f"Fetching videos between {START_DATE} and {END_DATE}...")
            video_titles = get_video_titles(YOUTUBE_API_KEY, uploads_playlist_id, start_date=START_DATE, end_date=END_DATE)
            print(f"Found {len(video_titles)} videos:")
        else:
            print(f"Fetching {MAX_VIDEOS} latest videos...")
            video_titles = get_video_titles(YOUTUBE_API_KEY, uploads_playlist_id, max_results=MAX_VIDEOS)
            print(f"Found {len(video_titles)} latest videos:")
        
        # Store video titles in a dataframe
        df = pd.DataFrame(video_titles, columns=["Title", "Published_At"])
        
        # Classify each title into categories
        df["Category"] = df["Title"].apply(classify_title)
        
        # Print the dataframe
        print(df)
        
        # Save the dataframe to a CSV file
        df.to_csv("classified_videos.csv", index=False)
        print("Data saved to 'classified_videos.csv'")
    
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    main()

Fetching Channel ID for handle '@CNBC-TV18'...
Channel ID for 'CNBC-TV18': UCmRbHAgG2k2vDUvb3xsEunQ
Uploads Playlist ID: UUmRbHAgG2k2vDUvb3xsEunQ
Fetching 10 latest videos...
Found 10 latest videos:
An error occurred: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
